In [ ]:
from google.colab import files
uploaded = files.upload()

Saving netflix_titles.csv to netflix_titles (1).csv


In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('/content/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:
df.info()
df.isnull().sum()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [ ]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Not Available')
df['country'] = df['country'].fillna('Unknown')

# drop rows where date is missing
df = df.dropna(subset=['date_added'])

In [ ]:
df['date_added'] = df['date_added'].str.strip()

df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

df = df.dropna(subset=['date_added'])

df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['day_added'] = df['date_added'].dt.day

In [ ]:
# FIX 1: raw string for regex
df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)

# FIX 2: handle NaN safely
df['duration_type'] = df['duration'].apply(
    lambda x: 'Minutes' if isinstance(x, str) and 'min' in x else 'Seasons'
)

In [ ]:
df['main_genre'] = df['listed_in'].apply(lambda x: x.split(',')[0])

In [ ]:
df['cast_count'] = df['cast'].apply(lambda x: len(x.split(',')))

In [ ]:
df['country'] = df['country'].str.split(',').str[0]

In [ ]:
df['type'] = df['type'].str.strip()

In [ ]:
!pip install textblob

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    return text

df['clean_description'] = df['description'].apply(clean_text)

In [ ]:
from collections import Counter

all_words = " ".join(df['clean_description']).split()

common_words = Counter(all_words).most_common(20)

print(common_words)

[('a', 11604), ('the', 8139), ('to', 6439), ('and', 6331), ('of', 5269), ('in', 4348), ('his', 3349), ('with', 2266), ('her', 2167), ('an', 1992), ('for', 1790), ('on', 1774), ('their', 1667), ('when', 1517), ('this', 1391), ('from', 1291), ('as', 1222), ('is', 1116), ('by', 1005), ('after', 993)]


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

filtered_words = [word for word in all_words if word not in ENGLISH_STOP_WORDS]

common_words = Counter(filtered_words).most_common(20)

print(common_words)

[('life', 772), ('young', 728), ('new', 699), ('family', 570), ('love', 496), ('man', 491), ('world', 490), ('friends', 466), ('woman', 452), ('series', 394), ('documentary', 361), ('finds', 313), ('school', 312), ('home', 306), ('help', 290), ('lives', 277), ('takes', 274), ('group', 263), ('years', 262), ('father', 238)]


In [ ]:
def extract_theme(desc):
    desc = desc.lower()

    if 'murder' in desc or 'kill' in desc:
        return 'Murder'
    elif 'love' in desc or 'romance' in desc:
        return 'Romance'
    elif 'war' in desc or 'battle' in desc:
        return 'Action'
    elif 'family' in desc:
        return 'Family'
    elif 'crime' in desc:
        return 'Crime'
    else:
        return 'Other'

df['theme'] = df['description'].apply(extract_theme)

In [ ]:
df['desc_length'] = df['description'].apply(len)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=50)

X = tfidf.fit_transform(df['clean_description'])

print(tfidf.get_feature_names_out())

['best' 'city' 'comedy' 'crime' 'daughter' 'death' 'documentary' 'family'
 'father' 'finds' 'friend' 'friends' 'gets' 'girl' 'group' 'help' 'hes'
 'high' 'home' 'life' 'lives' 'love' 'make' 'man' 'mother' 'murder'
 'mysterious' 'new' 'past' 'save' 'school' 'series' 'son' 'soon' 'special'
 'story' 'takes' 'team' 'teen' 'town' 'tries' 'true' 'war' 'way' 'wife'
 'woman' 'women' 'world' 'years' 'young']


In [ ]:
df['content_age'] = df['year_added'] - df['release_year']

In [ ]:
df['title_length'] = df['title'].apply(len)

In [ ]:
df['is_movie'] = df['type'].apply(lambda x: 1 if x == 'Movie' else 0)

In [ ]:
df.to_csv('netflix_cleaned.csv', index=False)

In [ ]:
from google.colab import files
files.download('netflix_cleaned.csv')

FileNotFoundError: Cannot find file: netflix_cleaned.csv